# NB04 — Particionado y árbol físico

Consume `dataset_etiquetado.parquet`; produce `dataset/` y actualiza el parquet según marco v2 §§3–7.

In [ ]:
# CONFIG único — D13-D17; RNG único: TA→NFS→PEO→balanceo→split.
import os,json,hashlib,shutil
from pathlib import Path
import numpy as np, pandas as pd
BASE_DIR=Path(os.environ.get('DERM_IAPP_BASE','/content/derm_iapp')); CONFIG={'BASE_DIR':BASE_DIR,'FULL_DATASET':os.environ.get('DERM_IAPP_FULL_DATASET','1')=='1','SEMILLA':2026,'SUBCONJUNTOS_POR_FAMILIA':int(os.environ.get('DERM_IAPP_SUBSETS','30')),'IMG_BCC_POR_SUB':int(os.environ.get('DERM_IAPP_BCC_SUB','15')),'IMG_NOBCC_POR_SUB':int(os.environ.get('DERM_IAPP_NOBCC_SUB','15')),'VAL_FRACTION':float(os.environ.get('DERM_IAPP_VAL_FRACTION','.20')),'TOLERANCIA_SPLIT':3,'ROTATIVO':['SCC','VASC','DF']}
CONFIG['CUOTA_NOBCC']=json.loads(os.environ.get('DERM_IAPP_CUOTA','{"NV":7,"MEL":4,"BKL":2,"AK":1,"ROTATIVO":1}')); FAMILIAS=['TA','NFS','PEO']
def clave(r): return r.lesion_id if pd.notna(r.lesion_id) and r.lesion_id else 'propias::'+r.image_id
def reservar_marco(pool,rng,config):
    usados=set(); filas=[]
    for familia in FAMILIAS:
        quemadas=set()
        for k in range(1,config['SUBCONJUNTOS_POR_FAMILIA']+1):
            clases=['BCC']*config['IMG_BCC_POR_SUB']+sum(([c]*n for c,n in config['CUOTA_NOBCC'].items() if c!='ROTATIVO'),[])+[config['ROTATIVO'][(k-1)%len(config['ROTATIVO'])]]*config['CUOTA_NOBCC'].get('ROTATIVO',0)
            for slot,c in enumerate(clases,1):
                q=pool[(pool.clase_9==c)&~pool.image_id.isin(usados)].sort_values('image_id'); q=q.iloc[rng.permutation(len(q))]; q=q[q.apply(lambda r:clave(r) not in quemadas,axis=1)]; assert len(q),f'Insuficientes imágenes {familia}/{k}/{c}'; r=q.iloc[0]; usados.add(r.image_id); quemadas.add(clave(r)); filas.append(dict(r, familia=familia,subconjunto_id=k,slot=slot))
    return pd.DataFrame(filas)
def balancear(pool,reserva,rng):
    rem=pool[~pool.image_id.isin(reserva.image_id)]; b=rem[rem.es_bcc]; no=rem[~rem.es_bcc]; n=len(b); cuotas={'NV':round(n*.48),'MEL':round(n*.26),'BKL':round(n*.12),'AK':round(n*.07)}; resto=n-sum(cuotas.values()); disp=no[no.clase_9.isin(['SCC','VASC','DF'])].clase_9.value_counts().sort_index(); assert len(disp) and disp.sum()>=resto,'No hay no-BCC rotativos suficientes'; base=disp/disp.sum()*resto; extra=base.astype(int)
    for c in (base-extra).sort_values(ascending=False).index[:resto-int(extra.sum())]: extra[c]+=1
    cuotas.update({c:int(extra.get(c,0)) for c in ['SCC','VASC','DF']}); toma=[]
    for c,nc in cuotas.items():
        q=no[no.clase_9==c].sort_values('image_id'); assert len(q)>=nc,f'Insuficientes no-BCC {c}'; toma.extend(q.iloc[rng.permutation(len(q))[:nc]].to_dict('records'))
    return pd.concat([b,pd.DataFrame(toma)],ignore_index=True)
def split_train_val(x,rng,config):
    x=x.copy(); x['_k']=x.apply(clave,axis=1); val=[]
    for clase in [True,False]:
        g=x[x.es_bcc==clase]; objetivo=round(len(g)*config['VAL_FRACTION']); actual=0; grupos=list(g.groupby('_k',sort=True))
        for i in rng.permutation(len(grupos)):
            z=grupos[i][1]
            if actual+len(z)<=objetivo+config['TOLERANCIA_SPLIT']: val+=z.image_id.tolist(); actual+=len(z)
        if not any(g.image_id.isin(val)): val.append(g.image_id.iloc[0])
    x['split']=np.where(x.image_id.isin(val),'VAL','TRAIN'); assert not (x.groupby('_k').split.nunique()>1).any(),'Lesión repartida entre TRAIN y VAL'; return x.drop(columns='_k')


In [ ]:
# PASOS 4–6 — árbol, manifiestos, invariantes y parquet canónico.
def _test(r): return f'TEST/{r.familia}/subconjunto_{r.subconjunto_id:02d}/{r.image_id}.jpg'
def _split(r): return f'{r.split}/{"BCC" if r.es_bcc else "NOBCC"}/{r.image_id}.jpg'
def _lesion(x): return '' if pd.isna(x) else str(x)
def _hash_sub(x): return hashlib.sha256('\n'.join(f'{r.image_id}:{r.sha256}' for r in x.sort_values('image_id').itertuples()).encode('utf-8')).hexdigest()
def crear_arbol(reserva,split,config):
    """Copia física idempotente; las etiquetas TEST viven solo en manifiestos."""
    ds=config['BASE_DIR']/'dataset'; cruda=config['BASE_DIR']/'data_cruda'
    for fam in FAMILIAS:
        for k in range(1,config['SUBCONJUNTOS_POR_FAMILIA']+1):
            d=ds/'TEST'/fam/f'subconjunto_{k:02d}'; d.mkdir(parents=True,exist_ok=True)
            for f in d.glob('*.jpg'): f.unlink()
    for parte in ['TRAIN','VAL']:
        for clase in ['BCC','NOBCC']:
            d=ds/parte/clase; d.mkdir(parents=True,exist_ok=True)
            for f in d.glob('*.jpg'): f.unlink()
    for r in reserva.itertuples(): shutil.copy2(cruda/r.ruta,ds/_test(r))
    for r in split.itertuples(): shutil.copy2(cruda/r.ruta,ds/_split(r))
    n=config['IMG_BCC_POR_SUB']+config['IMG_NOBCC_POR_SUB']
    for fam in FAMILIAS:
        for k in range(1,config['SUBCONJUNTOS_POR_FAMILIA']+1): assert len(list((ds/'TEST'/fam/f'subconjunto_{k:02d}').glob('*.jpg')))==n,f'TEST incompleto {fam}/{k}'
    for parte in ['TRAIN','VAL']:
        for clase,b in [('BCC',True),('NOBCC',False)]: assert len(list((ds/parte/clase).glob('*.jpg')))==int(((split.split==parte)&(split.es_bcc==b)).sum()),f'{parte}/{clase} incompleto'
def escribir_manifiestos(reserva,config,sha_entrada):
    ds=config['BASE_DIR']/'dataset'; cols=['image_id','familia','subconjunto_id','es_bcc','clase_9','lesion_id','origen','ruta','sha256']; todo=[]
    for fam in FAMILIAS:
        for k in range(1,config['SUBCONJUNTOS_POR_FAMILIA']+1):
            x=reserva[(reserva.familia==fam)&(reserva.subconjunto_id==k)].copy().sort_values('slot'); x['lesion_id']=x.lesion_id.map(_lesion); x['ruta']=x.apply(_test,axis=1); assert len(x)==config['IMG_BCC_POR_SUB']+config['IMG_NOBCC_POR_SUB']; x[cols].to_csv(ds/'TEST'/fam/f'subconjunto_{k:02d}'/f'subconjunto_{k:02d}.csv',index=False); x['sha256_subconjunto']=_hash_sub(x); todo.append(x[['familia','subconjunto_id','slot','image_id','es_bcc','clase_9','lesion_id','origen','ruta','sha256','sha256_subconjunto']])
    g=pd.concat(todo).sort_values(['familia','subconjunto_id','slot'],key=lambda s:pd.Categorical(s,FAMILIAS,ordered=True) if s.name=='familia' else s); g.to_csv(ds/'asignacion_global.csv',index=False)
    e={'pool_total':31469,'pool_bcc':4298,'pool_nobcc':27171,'unk_excluidas':2047,'duplicados_excluidos':50,'casi_duplicados_excluidos':3,'subconjuntos':90,'imagenes_por_subconjunto':30,'reserva_bcc':1350,'reserva_nobcc':1350,'remanente':28769,'balanceado_por_clase':2948,'no_usadas_nobcc':22873}
    o={k:None for k in ['pool_total','pool_bcc','pool_nobcc','reserva_unica_bcc','reserva_unica_nobcc','balanceado_bcc','balanceado_nobcc','train_bcc','train_nobcc','val_bcc','val_nobcc','desvio_val_bcc','desvio_val_nobcc','lesiones_mixtas_train_val','no_usadas_nobcc']}
    par={'semilla':2026,'val_fraction':config['VAL_FRACTION'],'familias':FAMILIAS,'subconjuntos_por_familia':config['SUBCONJUNTOS_POR_FAMILIA'],'img_bcc_por_subconjunto':config['IMG_BCC_POR_SUB'],'img_nobcc_por_subconjunto':config['IMG_NOBCC_POR_SUB'],'cuota_nobcc':config['CUOTA_NOBCC'],'orden_rotativo':config['ROTATIVO'],'full_dataset':config['FULL_DATASET'],'tamanos_entrada_esperados':{'ISIC_2019_Training_Metadata.csv':1214351,'ISIC_2019_Training_GroundTruth.csv':1291479,'ISIC_2019_Test_Metadata.csv':293638,'ISIC_2019_Test_GroundTruth.csv':464810,'ISIC_2019_Training_Input.zip':9771618190,'ISIC_2019_Test_Input.zip':3823247963},'conteos_esperados_i1_i10':e,'conteos_observados':o,'sha256_parquet_entrada':sha_entrada,'marco':'v2','notebook':'04_particionado_90_subconjuntos.ipynb','fecha':'2026-11-04'}
    (ds/'parametros.json').write_text(json.dumps(par,sort_keys=True,indent=2,ensure_ascii=False),encoding='utf-8'); (ds/'semilla.txt').write_text('2026',encoding='utf-8')
    return g

def verificar_invariantes(pool,reserva,bal,split,config,sha_entrada):
    """Verifica I1–I10, actualiza destinos/rutas y reescribe el parquet ordenado."""
    ds=config['BASE_DIR']/'dataset'; n=config['IMG_BCC_POR_SUB']+config['IMG_NOBCC_POR_SUB']; slots=len(FAMILIAS)*config['SUBCONJUNTOS_POR_FAMILIA']*n
    assert len(reserva)==slots and reserva.groupby(['familia','subconjunto_id']).size().eq(n).all(),'I3: subconjuntos inválidos'
    for _,x in reserva.groupby('familia'): assert not x.image_id.duplicated().any() and not x.apply(clave,axis=1).duplicated().any(),'I4: colisión intra-familia'
    assert reserva.image_id.nunique()==slots,'I5: reserva única distinta de slots'; rem=pool[~pool.image_id.isin(reserva.image_id)]; assert int(bal.es_bcc.sum())==int(rem.es_bcc.sum()) and int((~bal.es_bcc).sum())==int(bal.es_bcc.sum()),'I6: balanceo inválido'
    for b in [True,False]: assert int((split.es_bcc==b).sum())==int((bal.es_bcc==b).sum()),'I7: split no suma balanceado'
    kt=set(split[split.split=='TRAIN'].apply(clave,axis=1)); kv=set(split[split.split=='VAL'].apply(clave,axis=1)); assert not kt&kv,'I10: lesión compartida'
    final=pool.copy(); final['_origen']=final.ruta; final['destino_asignado']='no_usadas'
    for r in reserva.itertuples(): final.loc[final.image_id.eq(r.image_id),['destino_asignado','ruta']]=[f'TEST_{r.familia}',_test(r)]
    for r in split.itertuples(): final.loc[final.image_id.eq(r.image_id),['destino_asignado','ruta']]=[f'{r.split}_{"BCC" if r.es_bcc else "NOBCC"}',_split(r)]
    dominio={'TEST_TA','TEST_NFS','TEST_PEO','TRAIN_BCC','TRAIN_NOBCC','VAL_BCC','VAL_NOBCC','no_usadas'}; assert final.destino_asignado.isin(dominio).all() and not final.destino_asignado.eq('pendiente').any(),'I9: destino pendiente/inválido'; assert not final[final.destino_asignado.str.startswith('TEST')].image_id.isin(split.image_id).any(),'TEST en TRAIN/VAL'; assert final[final.destino_asignado.eq('no_usadas')].ruta.eq(final[final.destino_asignado.eq('no_usadas')]._origen).all(),'ruta no_usadas alterada'
    for fam in FAMILIAS:
        for k in range(1,config['SUBCONJUNTOS_POR_FAMILIA']+1):
            d=ds/'TEST'/fam/f'subconjunto_{k:02d}'; assert (d/f'subconjunto_{k:02d}.csv').exists() and len(list(d.glob('*.jpg')))==n,'manifiesto TEST incompleto'
    assert len(pd.read_csv(ds/'asignacion_global.csv'))==slots,'manifiesto global incompleto'
    if config['FULL_DATASET']:
        assert len(pool)==31469 and int(pool.es_bcc.sum())==4298 and int((~pool.es_bcc).sum())==27171,'I1'; assert int(reserva.es_bcc.sum())==1350 and int((~reserva.es_bcc).sum())==1350,'I5'; assert int(bal.es_bcc.sum())==2948 and int((~bal.es_bcc).sum())==2948,'I6'
        for b in [True,False]: assert abs(int(((split.split=='VAL')&(split.es_bcc==b)).sum())-590)<=3,'I7: desvío VAL > ±3'
        assert int(final[final.destino_asignado.eq('no_usadas')].es_bcc.sum())==0 and abs(int(final.destino_asignado.eq('no_usadas').sum())-22873)<=3,'I8'
    final=final.drop(columns='_origen').sort_values('image_id').reset_index(drop=True); final.to_parquet(config['BASE_DIR']/'dataset_etiquetado.parquet',index=False)
    par=json.loads((ds/'parametros.json').read_text(encoding='utf-8')); o=par['conteos_observados']; o.update({'pool_total':len(pool),'pool_bcc':int(pool.es_bcc.sum()),'pool_nobcc':int((~pool.es_bcc).sum()),'reserva_unica_bcc':int(reserva.es_bcc.sum()),'reserva_unica_nobcc':int((~reserva.es_bcc).sum()),'balanceado_bcc':int(bal.es_bcc.sum()),'balanceado_nobcc':int((~bal.es_bcc).sum()),'train_bcc':int(((split.split=='TRAIN')&split.es_bcc).sum()),'train_nobcc':int(((split.split=='TRAIN')&~split.es_bcc).sum()),'val_bcc':int(((split.split=='VAL')&split.es_bcc).sum()),'val_nobcc':int(((split.split=='VAL')&~split.es_bcc).sum()),'desvio_val_bcc':int(((split.split=='VAL')&split.es_bcc).sum())-round(int(bal.es_bcc.sum())*config['VAL_FRACTION']),'desvio_val_nobcc':int(((split.split=='VAL')&~split.es_bcc).sum())-round(int((~bal.es_bcc).sum())*config['VAL_FRACTION']),'lesiones_mixtas_train_val':len(kt&kv),'no_usadas_nobcc':int(final.destino_asignado.eq('no_usadas').sum())}); par['conteos_observados']=o; (ds/'parametros.json').write_text(json.dumps(par,sort_keys=True,indent=2,ensure_ascii=False),encoding='utf-8'); return final
def _asig(r,s):
    d={(x.familia,x.subconjunto_id,x.slot):x.image_id for x in r.itertuples()}; d.update({('SPLIT',x.image_id):x.split for x in s.itertuples()}); return d
def main():
    p=BASE_DIR/'dataset_etiquetado.parquet'; assert p.exists(),'ejecuta primero 03_limpieza_etiquetado.ipynb'; sha=hashlib.sha256(p.read_bytes()).hexdigest(); pool=pd.read_parquet(p); cols=['image_id','origen','clase_9','es_bcc','lesion_id','edad','sexo','sitio','ruta','sha256','destino_asignado']; assert list(pool.columns)==cols and set(pool.destino_asignado)=={'pendiente'}; assert sum(CONFIG['CUOTA_NOBCC'].values())==CONFIG['IMG_NOBCC_POR_SUB']
    def pasada():
        rng=np.random.default_rng(2026); r=reservar_marco(pool,rng,CONFIG); b=balancear(pool,r,rng); return r,b,split_train_val(b,rng,CONFIG)
    r,b,s=pasada(); r2,b2,s2=pasada(); assert _asig(r,s)==_asig(r2,s2),'Determinismo falló: las dos pasadas producen asignaciones distintas'
    crear_arbol(r,s,CONFIG); escribir_manifiestos(r,CONFIG,sha); final=verificar_invariantes(pool,r,b,s,CONFIG,sha); z=final.assign(_c=np.where(final.es_bcc,'BCC','no-BCC')).pivot_table(index='destino_asignado',columns='_c',aggfunc='size',fill_value=0); z['total']=z.sum(axis=1); print(z.reindex(columns=['BCC','no-BCC','total']).to_string()); return final
if __name__=='__main__': main()
